## Clinical Classification

In [ ]:
import lime
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import pickle
import random
import seaborn as sns
import tensorflow as tf

from keras.layers import Dense, Dropout, MaxPooling2D, Flatten, Conv2D, BatchNormalization
from keras.models import Sequential
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import classification_report, roc_curve, auc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow import keras
from tensorflow.keras.optimizers import Adam

### Loading & Preparing the X Datasets

Initially, the code loads the X training, validation, and test datasets stored as pickle files. The data is then examined to understand its structure and dimensions. Additional columns related to demographics and clinical information are added to the validation and test datasets to ensure consistency across all datasets. These columns are filled with zeros as placeholders. 

Next, clinical data columns relevant to Alzheimer's disease prediction are extracted from the merged datasets. These include various demographic attributes and clinical measurements such as age, education level, neuroimaging biomarkers, and cognitive assessment scores. The extracted clinical data is then separated into three subsets: training, validation, and test. 

Finally, the code prints the first few rows of the training dataset along with its shape and column names for verification purposes.

In [ ]:
# loading train, test, and validation datasets
X_train = pd.read_pickle(r"C:\Users\kishe\Documents\Year 3 Jupyter\X_train_merged.pkl")
X_val = pd.read_pickle(r"C:\Users\kishe\Documents\Year 3 Jupyter\X_val_merged.pkl")
X_test = pd.read_pickle(r"C:\Users\kishe\Documents\Year 3 Jupyter\X_test_merged.pkl")

print(type(X_train))
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

In [ ]:
# printing first row of dataset for analysis
row = X_train.iloc[0]
print(row)

In [ ]:
# columns to add to the X_val dataset which are missing
columns_to_add = ['PTETHCAT_Unknown', 'PTRACCAT_Asian', 'PTRACCAT_More than one', 'PTMARRY_Unknown', 'PTETHCAT_Not Hisp/Latino']

# adding new columns filled with zeros
for col in columns_to_add:
    X_val[col] = 0

print(X_val.head())

In [ ]:
# columns to add to the X_test dataset which are missing
columns_to_add = ['PTETHCAT_Unknown', 'PTRACCAT_Asian', 'PTMARRY_Unknown', 'PTETHCAT_Not Hisp/Latino']

# adding new columns filled with zeros
for col in columns_to_add:
    X_test[col] = 0

print(X_test.head())

In [ ]:
# extracting clinical data from the merged dataset and creating clinical train, val and test datasets
clinical_columns = ['AGE', 'PTEDUCAT', 'FDG', 'ABETA', 'TAU', 'PTAU', 'CDRSB', 'ADAS11', 'ADAS13', 'ADASQ4', 'MMSE', 
                'RAVLT_immediate', 'RAVLT_learning', 'RAVLT_forgetting', 'RAVLT_perc_forgetting', 'LDELTOTAL',
                'DIGITSCOR', 'TRABSCOR', 'FAQ', 'Ventricles', 'Hippocampus', 'WholeBrain', 'Entorhinal',
                'Fusiform', 'MidTemp', 'ICV', 'mPACCdigit', 'mPACCtrailsB', 'PTGENDER_Male', 'PTETHCAT_Not Hisp/Latino',
                'PTETHCAT_Unknown', 'PTRACCAT_Black', 
                'PTRACCAT_White', 'PTMARRY_Married','PTMARRY_Widowed', 'APOE4_1.0', 'APOE4_2.0'
                # 'PTRACCAT_Asian', 'PTMARRY_Never married', 'PTMARRY_Unknown', 'PTRACCAT_More than one'
                ]
X_train_clinical = X_train[clinical_columns]
X_val_clinical = X_val[clinical_columns]
X_test_clinical = X_test[clinical_columns]

print(X_train_clinical.head(10))

In [ ]:
print("Shape of X_train_clinical:", X_train_clinical.shape)
print(X_train_clinical.columns)

In [ ]:
row = X_train_clinical.iloc[0]
print(row)

### Loading & Preparing the Y Labels Datasets
The code extracts the diagnosis labels from the training, validation, and test datasets. These represent the clinical diagnosis of each subject and are stored in separate variables (`y_train`, `y_val`, and `y_test`). The labels are then flattened to 1D arrays and mapped to numerical values. 

A dictionary (`label_mapping`) is defined to map diagnostic categories ("AD" for Alzheimer's disease, "CN" for cognitively normal, and "MCI" for mild cognitive impairment) to numerical values (0, 1, and 2, respectively). The labels in the training, validation, and test sets are then replaced with their corresponding numerical values using list comprehensions and NumPy arrays.

In [ ]:
# extracting the diagnosis labels from the merged dataset
y_train = X_train["Diagnosis"]
y_val = X_val["Diagnosis"]
y_test = X_test["Diagnosis"]

# flattening to 1D array
y_train = y_train.values.ravel()
y_val = y_val.values.ravel() 
y_test = y_test.values.ravel()

print(type(y_train))
print(y_train[:20])
print(type(y_train))
print(y_train[:20])
print(y_train.shape)
print(y_val.shape)
print(y_test.shape)

In [ ]:
# encoding the categorical diagnosis labels with numerical mappings
label_mapping = {"AD": 0, "CN": 1, "MCI": 2}
y_train = np.array([label_mapping[label] for label in y_train])
y_val = np.array([label_mapping[label] for label in y_val])
y_test = np.array([label_mapping[label] for label in y_test])

### Training the FFN Classifier

Code evaluation loop developed with guidance and adapted from: https://github.com/rsinghlab/MADDi/blob/main/training/train_clinical.py

The network below is based on a model evaluation loop where a neural network model is trained and evaluated multiple times using different random seeds to assess its performance variability. 

First, a function `reset_random_seeds(seed)` is defined to ensure reproducibility by setting random seeds for various libraries like TensorFlow, NumPy, and Python's built-in `random` module. A list of random seeds (`seeds`) is then generated. For each seed in the list, the random seeds are reset using the `reset_random_seeds()` function, ensuring that each model training run is initialised with the same starting conditions. 

Within the loop, a feedforward neural network model is constructed consisting of several densely connected layers with rectified linear unit (ReLU) activation functions, batch normalisation layers, and dropout layers to prevent overfitting. The output layer uses the softmax activation function to produce probability distributions over the output classes.

The model is compiled with the Adam optimizer and sparse categorical cross-entropy loss function. Additionally, the sparse categorical accuracy metric is specified for evaluation.

The model is then trained using the training dataset (`X_train_clinical` and `y_train`) for a fixed number of epochs with early stopping criteria based on the validation dataset (`X_val_clinical` and `y_val`). 

After training, the model is evaluated on the test dataset (`X_test_clinical` and `y_test`). Metrics such as test loss and test accuracy are computed and printed for each run. The accuracy, precision, recall, and F1-score for each run are also calculated using the `classification_report` function from scikit-learn.

Finally, the average and standard deviation of the accuracy, precision, recall, and F1-score across all runs are computed and printed, along with the lists of individual metric values.

In [ ]:
def reset_random_seeds(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)  
    tf.random.set_seed(seed)  
    np.random.seed(seed)  
    random.seed(seed)  

# lists to store evaluation metrics
acc = []
f1 = []
precision = []
recall = []
val_accs = []

# generating random seeds for experiments
seeds = [42, 10, 53, 78, 20]

# looping through each seed
for seed in seeds:
    reset_random_seeds(seed)
    print("Seed:", seed)
    # defining the neural network model and adding layers
    model = Sequential()
    model.add(Dense(128, input_shape=(37,), activation="relu"))
    model.add(BatchNormalization())
    model.add(Dropout(0.5))
    model.add(Dense(64, activation="relu"))
    model.add(BatchNormalization())
    model.add(Dropout(0.3))
    model.add(Dense(50, activation="relu"))
    model.add(BatchNormalization())
    model.add(Dropout(0.2))
    model.add(Dense(3, activation="softmax"))

    # compiling the model
    model.compile(Adam(learning_rate=0.001), "sparse_categorical_crossentropy", metrics=["sparse_categorical_accuracy"])

    model.summary()

    # training and evaluating the model
    history = model.fit(X_train_clinical, y_train, epochs=250, validation_data=(X_val_clinical, y_val), batch_size=16, verbose=1) 
    score = model.evaluate(X_test_clinical, y_test, verbose=0)
    print(f'Test loss: {score[0]} / Test accuracy: {score[1]}')
    acc.append(score[1])

    # calculating validation accuracy
    val_accs.append(history.history['val_sparse_categorical_accuracy'])

    # making predictions on test data
    test_predictions = model.predict(X_test_clinical)
    test_label = to_categorical(y_test, 3)
    true_label = np.argmax(test_label, axis=1)
    predicted_label = np.argmax(test_predictions, axis=1)

    # calculating classification report
    cr = classification_report(true_label, predicted_label, output_dict=True)
    precision.append(cr["macro avg"]["precision"])
    recall.append(cr["macro avg"]["recall"])
    f1.append(cr["macro avg"]["f1-score"])

# printing validation accuracies for each seed
print("Validation Accuracies for Each Seed:")
for i, val_acc in enumerate(val_accs):
    print(f"Seed {seeds[i]}: {val_acc[-1]}")

print("Avg accuracy:", np.array(acc).mean())
print("Avg precision:", np.array(precision).mean())
print("Avg recall:", np.array(recall).mean())
print("Avg f1:", np.array(f1).mean())
print("Std accuracy:", np.array(acc).std())
print("Std precision:", np.array(precision).std())
print("Std recall:", np.array(recall).std())
print("Std f1:", np.array(f1).std())
print("Accuracies:", acc)
print("Precisions:", precision)
print("Recalls:", recall)
print("F1-scores:", f1)

## Explainable AI

### Yellowbrick
From: https://www.scikit-yb.org/en/latest/

Using the `FeatureCorrelation` visualiser from the Yellowbrick library to explore the relationship between features and the target variable. It calculates the mutual information between each feature and the target variable for classification tasks.

After fitting the visualiser with the training data, it displays a graph where features are sorted based on their mutual information with the target variable. This visualisation helps identify features that are informative for predicting the target.

In [ ]:
from yellowbrick.target import FeatureCorrelation

feature_names = list(X_train_clinical.columns)

# Instaniate the visualizer
visualizer = FeatureCorrelation(
    method='mutual_info-classification', feature_names=feature_names, sort=True
)

visualizer.fit(X_train_clinical, y_train)     
visualizer.show()      

The `Rank2D` visualiser is used to highlight pairwise feature correlations through Pearson correlation coefficients. The Pearson correlation gauges the linear connection between two variables, computing a score within the range of -1 to 1. A value of 1 signifies a perfect positive linear relationship, -1 denotes a perfect negative linear relationship, and 0 implies no linear relationship.

This visualiser calculates Pearson correlation coefficients across all feature pairs within the dataset and showcases them in a two-dimensional heatmap.

In [ ]:
from yellowbrick.features import Rank2D

visualizer = Rank2D(algorithm="pearson")
visualizer.fit_transform(X_train_clinical)
visualizer.show()

### LIME

Imports the LimeTabularExplainer module from the LIME library to provide local interpretable explanations for predictions made by the feedforward network trained on the clinical data. It starts by defining the class names for the target variable ('AD', 'CN', 'MCI') and retrieving the feature names from the test dataset. Then, it fits the LimeTabularExplainer on the training data set using the feature names and class names.

Next, it selects an instance from the test dataset for which explanations are desired and retrieves the corresponding ground truth label. The script then generates explanations for the chosen instance using the `LimeTabularExplainer.explain_instance` method, passing the instance values and the model's predict function. Finally, it displays the explanation in a notebook-friendly format using the `explanation.show_in_notebook()` method.

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

# getting the class and feature names
class_names = ['AD', 'CN', 'MCI']

feature_names = list(X_test_clinical.columns)

# fitting the Explainer on the training data set using the LimeTabularExplainer
explainer = LimeTabularExplainer(X_test_clinical.values, feature_names =     
                                 feature_names,
                                 class_names = class_names, 
                                 mode = 'classification')

In [ ]:
# choosing an instance from the test dataset for which to generate explanations
instance_idx = 6  

# getting instance and the corresponding ground truth label
instance = X_test_clinical.iloc[instance_idx]
true_label = class_names[y_test[instance_idx]]

# generating explanations for the chosen instance
explanation = explainer.explain_instance(instance.values, model.predict, num_features=len(feature_names))

explanation.show_in_notebook()

## SHAP

Creates an object named `explainer`. This object takes as input the predict method of the model and the training dataset. To ensure SHAP's model-agnostic nature, it perturbs the points within the training dataset and evaluates the model's response to these perturbations. 
This methodology aligns with LIME, which has been demonstrated to be a specific instance of the original SHAP methodology. The outcome is a statistical approximation of the SHAP values.

In [ ]:
import shap

feature_names = X_test_clinical.columns

explainer = shap.KernelExplainer(model.predict, X_train_clinical)

shap_values = explainer.shap_values(X_test_clinical, nsamples=100)

shap.summary_plot(shap_values, X_test_clinical, feature_names=feature_names)

In [ ]:
shap.initjs()

predicted_probabilities = model.predict(X_train_clinical)
base_value = np.mean(predicted_probabilities[:, 1]) 

shap.force_plot(base_value, shap_values[0], X_test_clinical.values, feature_names=feature_names)

In [ ]:
shap.summary_plot(shap_values, features=X_train_clinical, feature_names=feature_names)

In [ ]:
shap.summary_plot(shap_values[0], X_test_clinical.values, feature_names=feature_names)